In [ ]:
%pip install "transformers==4.57.6" "datasets==5.0.1" \
    "accelerate==1.15.0" "pythainlp==5.3.8" \
    "sentencepiece==0.2.1" --quiet

# Module M3 — สอนโมเดลอ่านใจจากข้อความไทย (fine-tune mBERT)

**โมดูลนี้รันบน Kaggle GPU** — สร้าง notebook บน [kaggle.com/code](https://www.kaggle.com/code) แล้วตั้งค่า **Settings → Accelerator: GPU T4 ×2** (ใช้ตัวเดียวพอ) และ **Internet: On** ก่อนกด Run All

- เซลล์แรกติดตั้ง dependencies ตาม pin ของ `requirements-kaggle.txt` (torch **ไม่ต้องติดตั้ง** — Kaggle ให้มาแล้ว)
- รันทั้งไฟล์ใช้เวลาราว **15–20 นาที บน T4** (เทรน ~8–12 นาที + ดาวน์โหลดโมเดลครั้งแรก ~1.8 GB รวมกัน)
- สถานะการตรวจ: โค้ดผ่าน lint และทดสอบ code path บน CPU แล้ว — **การรันจริงบน Kaggle T4 ต้องตรวจด้วยมือ 1 รอบก่อนวันอบรม** (ดู checklist ใน README)

**ที่มาและ license ของทุกอย่างที่โมดูลนี้ใช้**

| สิ่งที่ใช้ | ที่มา | license |
|---|---|---|
| Wisesight Sentiment (~26.7k ข้อความโซเชียลไทย: 21,628 train / 2,404 val / 2,671 test) | [HF: pythainlp/wisesight_sentiment](https://huggingface.co/datasets/pythainlp/wisesight_sentiment) · ต้นทาง [PyThaiNLP/wisesight-sentiment](https://github.com/PyThaiNLP/wisesight-sentiment) | **CC0 1.0** |
| mBERT — `google-bert/bert-base-multilingual-cased` (178M parameters, 104 ภาษา) | [HF: google-bert/bert-base-multilingual-cased](https://huggingface.co/google-bert/bert-base-multilingual-cased) | **Apache-2.0** |
| PyThaiNLP (ตัดคำภาษาไทย) | [github.com/PyThaiNLP/pythainlp](https://github.com/PyThaiNLP/pythainlp) | **Apache-2.0** |
| XLM-T zero-shot — `cardiffnlp/twitter-xlm-roberta-base-sentiment` | [HF model card](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment) | ไม่ระบุ license บน model card (โค้ด XLM-T: MIT) — ใช้เพื่อการสาธิตในห้องเรียน

## 0. โน้ตช่วย — คำศัพท์ที่ใช้ในโมดูลนี้ (สำหรับผู้ที่ไม่ได้เรียนคอร์ส DS)

| คำ | ความหมาย |
|---|---|
| **โมเดลภาษา (language model)** | โมเดลที่อ่านข้อความแล้วแปลงเป็นตัวเลข (เวกเตอร์) ที่จับ "ความหมาย" ได้ — เทรนมาแล้วบนข้อความมหาศาล |
| **fine-tune** | เอาโมเดลที่คนอื่นเทรนมาแล้ว มาเทรนต่อด้วยข้อมูลของเรา ให้ทำงานตรงโจทย์เรา (เหมือนการสอบเทียบเครื่องมือสำเร็จรูปให้ตรงกับชิ้นงานของเรา) |
| **loss** | ตัวเลขที่วัดว่าโมเดลพลาดมากแค่ไหน — **ยิ่งน้อยยิ่งดี** การเทรนคือการลดตัวเลขนี้ลงเรื่อย ๆ |
| **epoch** | หนึ่งรอบการอ่านข้อมูลเทรนทั้งชุด — เทรน 3 epochs = โมเดลอ่านข้อมูลเรา 3 รอบ |
| **batch** | จำนวนตัวอย่างที่ป้อนเข้าโมเดลพร้อมกันหนึ่งครั้ง (ที่นี่ 32 ข้อความ/ครั้ง) |
| **learning rate** | ขนาดก้าวที่โมเดล "ขยับ" ตัวเองทีละรอบ — ใหญ่ไปล้ม, เล็กไปช้า |
| **precision** | จากข้อความที่โมเดลบอกว่าเป็น class X → ถูกจริงกี่ % |
| **recall** | จากข้อความที่เป็น class X จริง → โมเดลจับได้กี่ % |

ถ้าจำได้จากโมดูล M1 (transfer learning) — โมดูลนี้คือ transfer learning เหมือนกัน แต่สลับจาก "ภาพ" เป็น "ข้อความไทย" และเทรน **ทั้งตัวโมเดล** (ไม่ freeze) เพราะข้อความไทยบนโซเชียลมีลักษณะเฉพาะที่ pre-train ไม่ค่อยครอบคลุม

In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np  # noqa: E402
import torch  # noqa: E402
import transformers  # noqa: E402

transformers.set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch",
      torch.__version__)
print("device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: ไม่มี GPU — บน CPU จะช้ามาก ให้รันบน Kaggle GPU")

## 1. ทำไม AI ภาษาไทย "ตัดคำ" ยาก

ภาษาอังกฤษเว้นวรรคระหว่างคำอยู่แล้ว แต่ **ภาษาไทยเขียนติดกันหมด** — คอมพิวเตอร์ต้องเดาเองว่าจะตัดตรงไหน และการเดาผิดเปลี่ยนความหมายได้จริง เช่น `ปลาทองแท้` (ปลา|ทองแท้ vs ปลาทอง|แท้) — นี่คือโจทย์ NLP ไทยคลาสสิกที่เรียกว่า **word segmentation**

โมดูลนี้เริ่มด้วย demo ตัดคำด้วย **PyThaiNLP** เพื่อให้เห็นปัญหา แล้วจะเห็นว่า mBERT แก้ปัญหานี้อย่างไร (คำตอบ: ไม่ตัดเป็นคำเลย แต่หั่นเป็นชิ้นเล็ก ๆ เรียก **wordpiece**)

In [ ]:
from pythainlp.tokenize import word_tokenize  # noqa: E402

t1 = "การประเมินความถูกต้องของเครื่องมือวัด"
t2 = "ราคาถูกจริงๆครับ ส่งไวมาก ของมาครบแต่กล่องยุบไปนิด"
print("split by space :", t1.split(" "))
print("word_tokenize  :", word_tokenize(t1))
print("split by space :", t2.split(" "))
print("word_tokenize  :", word_tokenize(t2))
print("longest engine :", word_tokenize(t1, engine="longest"))

สังเกตว่า `split(" ")` ทำอะไรไม่ได้เลย (ไทยไม่เว้นวรรค) — ส่วน word_tokenize ตัดได้ระดับคำแต่ก็ไม่สมบูรณ์ 100%

ต่อไปดูวิธีของ mBERT: หั่นข้อความเป็น **wordpiece** (ชิ้นย่อยของคำ จากคลัง ~119k ชิ้น ครอบคลุม 104 ภาษา) — ไม่ต้องรู้ "คำ" เลย ก็แทนข้อความเป็นตัวเลขได้

In [ ]:
from transformers import AutoTokenizer  # noqa: E402

CHECKPOINT = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print(tokenizer.tokenize("การสอบเทียบเครื่องมือวัด")[:14])

## 2. ชุดข้อมูล Wisesight Sentiment

ข้อความจริงจากโซเชียลมีเดียไทย (Facebook/Twitter) ที่ทีม Wisesight ตรวจให้ label ทีละข้อความ — โจทย์: **อ่านใจลูกค้าจากข้อความ** 4 class:

- `pos` — เชิงบวก · `neu` — ปกติ/เล่าเรื่อง · `neg` — เชิงลบ (บ่น/ร้องเรียน) · `q` — คำถาม

โจทย์แบบนี้เจอจริงในงานเมโทรโลยีเช่นกัน: อ่านความเห็นผู้ใช้เครื่องมือวัด, คัดกรองข้อร้องเรียน, สำรวจความพึงพอใจ

In [ ]:
from datasets import load_dataset  # noqa: E402

ds = load_dataset("pythainlp/wisesight_sentiment")
LABELS = ds["train"].features["category"].names
print(ds)
print("labels:", LABELS)

ดูสัดส่วน class และตัวอย่างข้อความจริง — สังเกตลักษณะภาษาโซเชียล: คำย่อ, อิโมจิ, สะกดผิด, เขียนติดกัน

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

y_train_all = np.array(ds["train"]["category"])
counts = np.bincount(y_train_all, minlength=len(LABELS))

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(LABELS, counts, color="teal")
ax.bar_label(bars)
ax.set_title("Wisesight train — จำนวนข้อความต่อ class")
plt.show()

for ci, cname in enumerate(LABELS):
    ex = [t for t, y in zip(ds["train"]["texts"], y_train_all)
          if y == ci][:2]
    print(f"[{cname}]")
    for t in ex:
        print("   ", t[:70])

สังเกตว่า class `neu` เยอะมาก และ `neg`/`q` น้อย — ข้อมูลจริงไม่เคยสมดุล นี่คือเหตุผลที่ต้องดู **precision/recall ต่อ class** ไม่ใช่แค่ accuracy เดียว

## 3. เตรียมข้อมูล

เพื่อให้ lab จบใน ~10 นาทีบน T4 เราเทรนบน **subsample 8,000 ข้อความ** (สุ่มแบบ stratified ตาม class, seed คงที่) — อยากใช้เต็ม 21,628 ข้อความก็แก้ `N_TRAIN = 21_628` แล้วรันใหม่ได้ (ผลดีขึ้นเล็กน้อย ใช้เวลา ~2.5 เท่า)

นอกจากนี้เราหั่น train เล็ก ๆ 1,000 ข้อความ (`train_probe`) ไว้ดู loss ฝั่ง train ราย epoch ด้วย

In [ ]:
N_TRAIN = 8_000
N_EPOCHS = 3

In [ ]:
from sklearn.model_selection import train_test_split  # noqa: E402

# subsample แบบ stratified ตาม class — จบเร็วขึ้นแต่สัดส่วนคลาสเดิม
y_all = np.array(ds["train"]["category"])
idx_sub, _ = train_test_split(
    np.arange(len(y_all)),
    train_size=N_TRAIN,
    stratify=y_all,
    random_state=SEED,
)
idx_probe, _ = train_test_split(
    np.arange(len(y_all)),
    train_size=1_000,
    stratify=y_all,
    random_state=SEED,
)

splits = {
    "train": ds["train"].select(idx_sub),
    "train_probe": ds["train"].select(idx_probe),
    "val": ds["validation"],
    "test": ds["test"],
}


def tokenize(batch):
    return tokenizer(batch["texts"], truncation=True, max_length=64)


tok = {}
for name, d in splits.items():
    d = d.rename_column("category", "labels")
    tok[name] = d.map(tokenize, batched=True)
print({name: len(d) for name, d in tok.items()})

## 4. Fine-tune mBERT — เห็น loss ลดลงราย epoch

ตั้งค่าสำคัญ (ค่ากลางที่ใช้กันทั่วไปสำหรับ fine-tune BERT):

| พารามิเตอร์ | ค่า | เหตุผล |
|---|---|---|
| `learning_rate` | 2e-5 | fine-tune ใช้ก้าวเล็กกว่าเทรนจากศูนย์มาก |
| `per_device_train_batch_size` | 32 | พอดี ๆ กับหน่วยความจำ T4 (16 GB) ที่ seq ยาว 64 |
| `num_train_epochs` | 3 | เห็น loss ลด 3 จุด — เพียงพอสำหรับการสอน |
| `fp16` | บน GPU | เร็วขึ้นราว 2 เท่าบน T4 ด้วย arithmetic 16 บิต |
| `max_length` | 64 tokens | ข้อความโซเชียลสั้น — ครอบคลุมเกือบทั้งหมด |
| `eval_strategy="epoch"` | — | วัดผลทุกจบ epoch → ได้กราฟ loss ราย epoch |

เซลล์ถัดไปดาวน์โหลด mBERT ครั้งแรก ~700 MB (รอสักครู่) แล้วติดตั้ง "หัว" ใหม่ (ชั้น linear 4 ทางออก) ต่อท้าย — ตอนเริ่มหัวนี้สุ่มเปล่า ๆ นั่นเองที่ทำให้ loss เริ่มต้นสูง

In [ ]:
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

id2label = {i: name for i, name in enumerate(LABELS)}
label2id = {name: i for i, name in enumerate(LABELS)}
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="m3_wisesight",
    seed=SEED,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=N_EPOCHS,
    weight_decay=0.01,
    fp16=DEVICE == "cuda",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}


trainer = Trainer(
    model,
    args,
    train_dataset=tok["train"],
    eval_dataset={
        "train_probe": tok["train_probe"],
        "val": tok["val"],
    },
    data_collator=DataCollatorWithPadding(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

กดรัน — แถบ progress แนวนอนหนึ่งแถบ = 1 epoch เทรนจะพิมพ์ loss ท้ายแต่ละ epoch ใช้เวลาราว 8–12 นาทีบน T4

In [ ]:
trainer.train()

## 4.1 อ่าน loss ราย epoch

`trainer.state.log_history` เก็บทุกตัวเลขที่โมเดลรายงานระหว่างเทรน — เราดึงมาพล็อต:

- **train loss** — ค่าเฉลี่ย loss ตอนเทรนในแต่ละ epoch (ควรลด)
- **train_probe / val loss** — loss ตอน "สอบ" ด้วยข้อมูลที่ไม่ได้ใช้เทรน (ควรลดตาม; ถ้า val ลดแล้วเริ่ม **ขึ้น** = เริ่ม overfit)

In [ ]:
history = trainer.state.log_history
epochs = [h["epoch"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]
probe_loss = [
    h["eval_train_probe_loss"] for h in history
    if "eval_train_probe_loss" in h
]
val_loss = [
    h["eval_val_loss"] for h in history if "eval_val_loss" in h
]
val_acc = [
    h["eval_val_accuracy"] for h in history
    if "eval_val_accuracy" in h
]
print("train loss ราย epoch :", [round(x, 4) for x in train_loss])
print("val loss   ราย epoch :", [round(x, 4) for x in val_loss])
print("val accuracy ราย epoch:", [round(x, 4) for x in val_acc])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, train_loss, "o-", label="train loss")
ax.plot(epochs, probe_loss, "s--", label="train_probe loss")
ax.plot(epochs, val_loss, "^--", label="val loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("loss ลดลงราย epoch ระหว่าง fine-tune")
ax.legend()
plt.show()

**อ่านกราฟ:** loss ต้องลดลงราย epoch — ถ้า val loss ลดช้ากว่า train loss มาก หรือเริ่มเบนขึ้น คือสัญญาณ overfit (แก้ด้วยข้อมูลมากขึ้น / epochs น้อยลง)

## 5. วัดผลบน test set — precision/recall ต่อ class

accuracy เดียวไม่พอ เพราะ class ไม่สมดุล: โมเดลที่ตอบ `neu` ไปหมดก็ได้ accuracy ~54% แต่ไม่มีประโยชน์ เราจึงดูราย class ด้วย `classification_report`

In [ ]:
from sklearn.metrics import classification_report  # noqa: E402

pred_out = trainer.predict(tok["test"])
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = pred_out.label_ids
texts_test = ds["test"]["texts"]
print(classification_report(
    y_true, y_pred,
    labels=range(len(LABELS)), target_names=LABELS, digits=3,
    zero_division=0,
))

**อ่านตาราง:**

- `precision` (ของ class X) = ทุกข้อความที่โมเดลตอบ X → ถูกจริงกี่ส่วน
- `recall` (ของ class X) = ทุกข้อความที่เป็น X จริง → โมเดลจับได้กี่ส่วน
- `macro avg` = ค่าเฉลี่ยทุก class เท่า ๆ กัน (ทำให้ class เล็กมีน้ำหนัก) · `weighted avg` = ถ่วงน้ำหนักด้วยจำนวนตัวอย่าง

พล็อตเป็นกราฟเพื่อเห็น class ที่โมเดลอ่อนทันที:

In [ ]:
from sklearn.metrics import precision_recall_fscore_support  # noqa: E402

prec, rec, _, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=range(len(LABELS)), zero_division=0,
)
x = np.arange(len(LABELS))
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w / 2, prec, w, label="precision", color="teal")
b2 = ax.bar(x + w / 2, rec, w, label="recall", color="coral")
ax.bar_label(b1, fmt="%.2f")
ax.bar_label(b2, fmt="%.2f")
ax.set_xticks(x, LABELS)
ax.set_ylim(0, 1.0)
ax.set_title("precision / recall ต่อ class (test)")
ax.legend()
plt.show()

## 6. Error inspection — โมเดลพลาดอะไร

ตัวเลขบอกว่าพลาดเท่าไร แต่ไม่บอกว่า **พลาดเพราะอะไร** — ต้องแกะดูตัวอย่างที่ผิดจริงต่อ class นี่คือสิ่งที่นักวิจัยต้องทำเสมอก่อนเชื่อตัวเลข

In [ ]:
for ci, cname in enumerate(LABELS):
    wrong = [
        i for i in range(len(y_true))
        if y_true[i] == ci and y_pred[i] != ci
    ]
    n_true = int((y_true == ci).sum())
    print(f"[{cname}] ผิด {len(wrong)} จาก {n_true} ตัวอย่าง")
    for i in wrong[:3]:
        print(f"   ({cname} -> {LABELS[y_pred[i]]})",
              texts_test[i][:70].replace("\n", " "))
    print()

**ประเด็นที่มักเจอกับ Wisesight:**

- `neg` ที่ถูกตอบเป็น `neu` — การบ่นแบบไทย ๆ ("ของมาช้านิดหน่อย") ไม่มีคำชัด โมเดลจับ tone ได้ยาก
- `q` กับ `neu` สับสนกันเยอะ — คำถามบางข้อไม่มีเครื่องหมาย ? และการเล่าเรื่องก็ดูเหมือนคำถาม
- ข้อความสั้นมาก / อิโมจิเป็นหลัก — บริบทน้อยเกินไปจริง ๆ (label ของมนุษย์ก็ตีความต่างกันได้)

## 7. Demo 5 นาที — embedding + kNN

นอกจากการจำแนก โมเดลภาษายังให้ **embedding** — เวกเตอร์ที่ข้อความ "ใกล้เคียงกัน" (ความหมาย/อารมณ์) อยู่ใกล้กันในปริภูมิตัวเลข ใช้ kNN (หาเพื่อนบ้านใกล้สุด) แล้วข้อความใหม่ก็หา "ข้อความเดิมที่เหมือนสุด" ได้ทันทีโดยไม่ต้องเทรนเพิ่ม — เครื่องมือจริงสำหรับงานค้นข้อความคล้ายกันในฐานข้อมูลเมโทรโลยี

เราดึงเวกเตอร์จากตำแหน่ง `[CLS]` (token แรก) ของชั้นสุดท้าย ซึ่งพอ fine-tune แล้วจะเก็บข้อมูลอารมณ์ของข้อความไว้ด้วย

In [ ]:
from sklearn.decomposition import PCA  # noqa: E402

rng = np.random.default_rng(SEED)
emb_idx = rng.choice(len(y_true), size=500, replace=False)
emb_texts = [texts_test[i] for i in emb_idx]
emb_labels = [LABELS[y_true[i]] for i in emb_idx]


MODEL_DEV = next(trainer.model.parameters()).device


@torch.no_grad()
def embed(texts):
    """เวกเตอร์ [CLS] จากชั้นสุดท้ายของโมเดลที่เทรนแล้ว."""
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    ).to(MODEL_DEV)
    out = trainer.model(**enc, output_hidden_states=True)
    cls = out.hidden_states[-1][:, 0]
    return cls.float().cpu().numpy()


E = embed(emb_texts)
pca = PCA(n_components=2)
xy = pca.fit_transform(E)

fig, ax = plt.subplots(figsize=(6, 5))
for ci, cname in enumerate(LABELS):
    m = [j for j, y in enumerate(emb_labels) if y == cname]
    ax.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.5, label=cname)
ax.set_title("embedding (CLS) ของ 500 ข้อความ — PCA 2D")
ax.legend()
plt.show()

**อ่าน scatter:** จุดเริ่มแยกเป็นกลุ่มตาม class ได้บ้างแล้ว (โดยเฉพาะ pos กับ neu) — และ kNN หาเพื่อนบ้านที่ "อารมณ์คล้ายกัน" ได้

In [ ]:
from sklearn.neighbors import NearestNeighbors  # noqa: E402

nn = NearestNeighbors(n_neighbors=6).fit(E)
# ตัวอย่างมักมี neg เสมอ (500 ตัวอย่าง) — สำรอง fallback ไว้เผื่อชุดเล็ก
neg_pool = np.where(np.array(emb_labels) == "neg")[0]
q_pos = int(neg_pool[0]) if len(neg_pool) else 0
dist, nbr_idx = nn.kneighbors(E[[q_pos]])
print("query:", emb_texts[q_pos][:70], f"[{emb_labels[q_pos]}]")
for rank in range(1, len(nbr_idx[0])):
    j = nbr_idx[0][rank]
    print(f"   #{rank} dist={dist[0][rank]:.3f} [{emb_labels[j]}]",
          emb_texts[j][:55].replace("\n", " "))

## 8. Zero-shot vs supervised

`XLM-T` คือโมเดล sentiment ที่เทรนบนทวีต 8 ภาษา (อาหรับ/อังกฤษ/ฝรั่งเศส/... **ไม่มีไทย**) — การใช้กับข้อความไทยเลยคือ **zero-shot**: ใช้โมเดลโดยไม่เทรนด้วยข้อมูลเป้าหมายเลย

เทียบกันบนตัวอย่าง 150 ข้อความ (เฉพาะ 3 class เพราะ zero-shot ไม่มีคำตอบสำหรับ `q`):

In [ ]:
ZS_CHECKPOINT = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
zs_tok = AutoTokenizer.from_pretrained(ZS_CHECKPOINT)
zs_model = AutoModelForSequenceClassification.from_pretrained(
    ZS_CHECKPOINT,
).to(DEVICE)

# XLM-T ทำนาย 3 class: 0=negative 1=neutral 2=positive
# Wisesight เรียง 0=pos 1=neu 2=neg 3=q — ตารางแปลงดังนี้
ZS_MAP = {0: 2, 1: 1, 2: 0}


@torch.no_grad()
def zero_shot(texts):
    """ทำนายด้วย XLM-T แล้วแปลง id ให้ตรงลำดับ LABELS ของ Wisesight."""
    logits = []
    for start in range(0, len(texts), 32):
        enc = zs_tok(
            texts[start:start + 32],
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt",
        ).to(DEVICE)
        out = zs_model(**enc)
        logits.append(out.logits.float().cpu())
    return torch.cat(logits).argmax(-1).numpy()


# เทียบเฉพาะ 3 class (q ไม่มีคำตอบ zero-shot) บนตัวอย่าง 150 ข้อความ
mask3 = y_true != 3
rng_zs = np.random.default_rng(SEED)
zs_idx = rng_zs.choice(np.where(mask3)[0], size=150, replace=False)

zs_pred = np.array([
    ZS_MAP[p] for p in zero_shot([texts_test[i] for i in zs_idx])
])
truth3 = y_true[zs_idx]
zs_acc = float((zs_pred == truth3).mean())
ft_acc = float((y_pred[zs_idx] == truth3).mean())
print(f"zero-shot  XLM-T (ไม่เคยเห็นไทย): {zs_acc:.3f}")
print(f"supervised mBERT (เทรนเอง {N_TRAIN} ข้อความ): {ft_acc:.3f}")

**สรุป zero-shot vs supervised:**

| | zero-shot (XLM-T) | supervised (mBERT ที่เทรนเอง) |
|---|---|---|
| ข้อมูลที่ต้องมี | ไม่ต้องมี | ต้องมี label (ที่นี่ 8k ข้อความ) |
| ความแม่นบนโจทย์ไทย | ต่ำกว่ามาก — ไทยเป็น out-of-domain | สูงกว่าชัดเจน |
| เวลา/ต้นทุน | ได้คำตอบทันที | ต้องเทรน + คน label |
| ใช้เมื่อไร | ต้องการคำตอบแรก ๆ เร็ว ๆ, ไม่มี label | มี label หลักพันขึ้นไป และต้องการความแม่น |

โมเดล multilingual pretrain ทำให้ zero-shot ข้ามภาษา "ใช้ได้บ้าง" แต่กับภาษาไทยโซเชียลของจริง supervised ชนะขาด — นี่คือเหตุผลที่หลักสูตรนี้สอนวงจร "ข้อมูล → เทรน → วัดผล"

## 9. สรุปโมดูล

- ภาษาไทยไม่เว้นวรรค → word segmentation เป็นโจทย์เก่าแก่; **โมเดล transformer ข้ามปัญหานี้ด้วย wordpiece**
- **fine-tune mBERT** บน Wisesight: loss ลดราย epoch และต้องอ่าน per-class precision/recall ทั้งสองค่า เพราะ class ไม่สมดุล
- **error inspection** บอกเรื่องที่ตัวเลขไม่บอก — ทำเป็นนิสัยก่อนเชื่อ metric
- **embedding + kNN** = จุดเริ่มของระบบค้นข้อความคล้ายกัน
- **zero-shot ใช้ได้แต่แพ้ supervised** เมื่อมี label ให้เทรน

ต่อ: `exercise.ipynb` — แล็บของคุณ (ตัดคำด้วยประโยคของตัวเอง, หา class ที่โมเดลอ่อน, error inspection, kNN และ zero-shot ด้วยมือ)